## **Despliegue de ETL utilizando SPARK SCALA**

### **Diagrama ETL**

<center><img src="https://i.postimg.cc/xjGY4vvD/p89.png"></center>

### **`Paso 1` - Crear directorio en el contenedor "namenode" que albergara los archivos de entrada que utilizaremos para el despliegue del procesamiento SPARK**

In [ ]:
# Lo hacemos en el contenedor "namenode"
C:\Users\Alfonso\Documents\ecosistema-hadoop>docker exec -it namenode bash
root@namenode:/#
root@namenode:/# pwd
/
root@namenode:/#
root@namenode:/# mkdir -p /hadoop-data/proyecto_3/archivos
root@namenode:/#
root@namenode:/# cd hadoop-data
root@namenode:/#
root@namenode:/hadoop-data# ls
proyecto_3
root@namenode:/hadoop-data# cd proyecto_3
root@namenode:/hadoop-data/proyecto_3#
root@namenode:/hadoop-data/proyecto_3# ls
archivos
root@namenode:/hadoop-data/proyecto_3# cd archivos
root@namenode:/hadoop-data/proyecto_3/archivos#
root@namenode:/hadoop-data/proyecto_3/archivos# ls
root@namenode:/hadoop-data/proyecto_3/archivos# exit
exit

What's next?
  Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug namenode
  Learn more at https://docs.docker.com/go/debug-cli/

C:\Users\Alfonso\Documents\ecosistema-hadoop>

### **`Paso 2` - Copiar archivos de entrada al directorio recién creado**

In [ ]:
docker cp RIESGO_CREDITICIO.csv namenode:/hadoop-data/proyecto_3/archivos
docker cp transacciones_bancarias.json namenode:/hadoop-data/proyecto_3/archivos
docker cp deploy_directorios.sh namenode:/hadoop-data/proyecto_3/archivos

In [ ]:
C:\Users\Alfonso\Documents\ecosistema-hadoop>cd material

C:\Users\Alfonso\Documents\ecosistema-hadoop\material>cd proyecto_3

C:\Users\Alfonso\Documents\ecosistema-hadoop\material\proyecto_3>docker cp RIESGO_CREDITICIO.csv namenode:/hadoop-data/proyecto_3/archivos
Successfully copied 3.58kB to namenode:/hadoop-data/proyecto_3/archivos

C:\Users\Alfonso\Documents\ecosistema-hadoop\material\proyecto_3>docker cp transacciones_bancarias.json namenode:/hadoop-data/proyecto_3/archivos
Successfully copied 4.49MB to namenode:/hadoop-data/proyecto_3/archivos

C:\Users\Alfonso\Documents\ecosistema-hadoop\material\proyecto_3>docker cp deploy_directorios.sh namenode:/hadoop-data/proyecto_3/archivos
Successfully copied 4.1kB to namenode:/hadoop-data/proyecto_3/archivos

C:\Users\Alfonso\Documents\ecosistema-hadoop\material\proyecto_3>docker exec -it namenode bash
root@namenode:/#
root@namenode:/# cd /hadoop-data/proyecto_3/archivos
root@namenode:/hadoop-data/proyecto_3/archivos#
root@namenode:/hadoop-data/proyecto_3/archivos# ls -l
root@namenode:/hadoop-data/proyecto_3/archivos# ls -l
total 4396
-rwxr-xr-x 1 root root    1658 Nov  8  2021 RIESGO_CREDITICIO.csv
-rwxr-xr-x 1 root root    2251 Jun 28 18:42 deploy_directorios.sh
-rwxr-xr-x 1 root root 4491352 Nov 22  2021 transacciones_bancarias.json
root@namenode:/hadoop-data/proyecto_3/archivos#
root@namenode:/hadoop-data/proyecto_3/archivos#

### **`Paso 3` - Despliegue de directorios en HDFS**

In [ ]:
sh deploy_directorios.sh proyectos proyecto_3

In [ ]:
C:\Users\Alfonso\Documents\ecosistema-hadoop\material\proyecto_3>docker exec -it namenode bash
root@namenode:/#
root@namenode:/# cd hadoop-data/proyecto_3/archivos
root@namenode:/hadoop-data/proyecto_3/archivos#
root@namenode:/hadoop-data/proyecto_3/archivos# sh deploy_directorios.sh proyectos proyecto_3
Eliminando carpeta raiz...
Creando directorio para almacenar archivos de entrada...
Creando directorio para almacenar archivos de salida...
Subiendo archivos de entrada...
root@namenode:/hadoop-data/proyecto_3/archivos# hdfs dfs -ls /user/proyectos/proyecto_3
Found 2 items
drwxr-xr-x   - root supergroup          0 2024-06-27 18:24 /user/proyectos/proyecto_3/input
drwxr-xr-x   - root supergroup          0 2024-06-27 18:24 /user/proyectos/proyecto_3/output
root@namenode:/hadoop-data/proyecto_3/archivos#
root@namenode:/hadoop-data/proyecto_3/archivos#

### **`Paso 4` - Crear directorio en el contenedor "spark-master" para almacenar el archivo "_build.sbt_", "_plugins.sbt_" y la ruta que albergara el archivo _.jar_**

Documentación compilación un script spark scala con **sbt**: https://spark.apache.org/docs/latest/quick-start.html#self-contained-applications

In [ ]:
mkdir -p /hadoop-data/proyecto_3/src/main/scala
mkdir -p /hadoop-data/proyecto_3/project

In [ ]:
C:\Users\Alfonso\Documents\ecosistema-hadoop>docker exec -it spark-master bash
root@spark-master:/#
root@spark-master:/# cd hadoop-data
root@spark-master:/hadoop-data# cd proyecto_3
root@spark-master:/hadoop-data/proyecto_3# ls -l
total 8
drwxr-xr-x 2 root root 4096 Jun 29 16:49 project
drwxr-xr-x 3 root root 4096 Jun 29 16:48 src
root@spark-master:/hadoop-data/proyecto_3#
root@spark-master:/hadoop-data/proyecto_3# cd src
root@spark-master:/hadoop-data/proyecto_3/src#
root@spark-master:/hadoop-data/proyecto_3/src# cd main
root@spark-master:/hadoop-data/proyecto_3/src/main#
root@spark-master:/hadoop-data/proyecto_3/src/main# cd scala
root@spark-master:/hadoop-data/proyecto_3/src/main/scala#
root@spark-master:/hadoop-data/proyecto_3/src/main/scala# exit
exit

What's next?
  Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug spark-master
  Learn more at https://docs.docker.com/go/debug-cli/

C:\Users\Alfonso\Documents\ecosistema-hadoop\material\proyecto_3>

### **`Paso 5` - Creamos los archivos "_build.sbt_" y "_plugins.sbt_""**

##### **Explicación modificaciones**

Al tratar de lanzar el comando **sbt package** me devolvia el error:
```text
Caused by: java.lang.BootstrapMethodError: java.lang.NoClassDefFoundError: scala/runtime/LambdaDeserialize
        at Main$.$deserializeLambda$(script_spark.scala)
        ... 78 more
Caused by: java.lang.NoClassDefFoundError: scala/runtime/LambdaDeserialize
        ... 79 more
```
El error **java.lang.BootstrapMethodError** junto con **java.lang.NoClassDefFoundError: scala/runtime/LambdaDeserialize** indica claramente que Spark está intentando deserializar un lambda (función anónima) y no puede encontrar la clase necesaria LambdaDeserialize en tiempo de ejecución. Esto suele ser un problema de configuración o de disponibilidad de clases en el entorno de ejecución de Spark.

Por tanto, una solución es utilizar **sbt assembly**:

- **sbt assembly**: Este comando genera un archivo JAR "gordo" (fat JAR) que incluye todas las dependencias necesarias para ejecutar tu aplicación Spark de manera independiente. Es útil cuando deseas crear un único archivo JAR que contenga todas las clases y dependencias, lo que facilita la distribución y la ejecución de la aplicación en un clúster de Spark.

Si estamos desarrollando una aplicación Spark que planeas ejecutar directamente en un clúster de Spark (por ejemplo, con spark-submit), generalmente usarías *sbt assembly*. **Esto asegura que todas las dependencias necesarias estén incluidas en el JAR para evitar problemas de clase no encontrada durante la ejecución**.

En resumen, **sbt assembly** es comúnmente utilizado para proyectos Spark para asegurar la portabilidad y la ejecución correcta de la aplicación en un clúster de Spark, mientras que **sbt package** es más adecuado cuando solo necesitas un JAR básico de tu aplicación sin incluir dependencias externas.

1. **Ubicación del Archivo `project/plugins.sbt`**:
   - En tu proyecto de SBT, debes tener un directorio llamado `project`. Dentro de este directorio es donde resides los archivos de configuración específicos del proyecto, incluyendo `plugins.sbt`.
   - Si aún no tienes un archivo `project/plugins.sbt`, créalo. Este archivo es donde se definen los plugins que deseas utilizar en tu proyecto de SBT.

2. **Agregar la Línea al Archivo `project/plugins.sbt`**:
   - Abre o crea el archivo `project/plugins.sbt`.
   - Agrega la línea siguiente línea al archivo:

     ```scala
     addSbtPlugin("com.eed3si9n" % "sbt-assembly" % "0.14.10")
     ```

     Esta línea indica a SBT que descargue e incluya el plugin `sbt-assembly` en tu proyecto, con la versión especificada (`0.14.10` en este caso).

3. **Ejemplo de `project/plugins.sbt` Completo**:
   Tu archivo `project/plugins.sbt` debería verse así después de agregar la línea del plugin `sbt-assembly`:

   ```scala
   // Archivo: project/plugins.sbt

   // Agregar plugin sbt-assembly
   addSbtPlugin("com.eed3si9n" % "sbt-assembly" % "0.14.10")
   ```

4. **Modificar `build.sbt`**:
   - Después de agregar el plugin en `project/plugins.sbt`, asegúrate de tener la configuración necesaria en `build.sbt` para usar `sbt-assembly` como se mencionó anteriormente. Es decir, deberías tener algo como:

     ```scala
     // Archivo: build.sbt

     name := "etl-spark-scala"

     version := "1.0"

     scalaVersion := "2.12.4"

     // Dependencias de Spark
     libraryDependencies += "org.apache.spark" %% "spark-core" % "2.4.5"
     libraryDependencies += "org.apache.spark" %% "spark-sql" % "2.4.5"

     // Configuración para el ensamblado (assembly)
     // Define el nombre del archivo JAR resultante
     assemblyJarName := "etl-spark-scala-assembly.jar"

     // Estrategia de merge para el ensamblado (assembly)
     assemblyMergeStrategy in assembly := {
     case PathList("META-INF", xs @ _*) => MergeStrategy.discard
     case x => MergeStrategy.first
     }

Con estos pasos, deberías tener configurado correctamente el plugin `sbt-assembly` en tu proyecto de SBT. Asegúrate de ejecutar `sbt assembly` desde la raíz de tu proyecto después de realizar estos cambios para ensamblar tu aplicación Spark con todas las dependencias incluidas en un archivo JAR.

##### **`plugins.sbt`**

In [ ]:
// Agregar plugin sbt-assembly
addSbtPlugin("com.eed3si9n" % "sbt-assembly" % "0.14.10")

##### **`build.sbt`**

Repositorio Maven para Spark: https://mvnrepository.com/search?q=org.apache.spark

In [ ]:
// Archivo: build.sbt

name := "etl-spark-scala"

version := "1.0"

scalaVersion := "2.12.4"

// Dependencias de Spark
libraryDependencies += "org.apache.spark" %% "spark-core" % "2.4.5"
libraryDependencies += "org.apache.spark" %% "spark-sql" % "2.4.5"

// Configuración para el ensamblado (assembly)
// Define el nombre del archivo JAR resultante
assemblyJarName := "etl-spark-scala-assembly.jar"

// Estrategia de merge para el ensamblado (assembly)
assemblyMergeStrategy in assembly := {
  case PathList("META-INF", xs @ _*) => MergeStrategy.discard
  case x => MergeStrategy.first 
}

In [ ]:
# Nombre de nuestro archivo .jar
name := "etl-spark-scala"

version := "1.0"

# La version debe ir de acuerdo a la version que tengamos de Scala
scalaVersion := "2.12.4"

# Las librerias-dependencias las obtenemos del Repositorio de Maven y deben ir de acuerdo a la version que tengamos de SPARK
libraryDependencies += "org.apache.spark" %% "spark-core" % "2.4.5"
libraryDependencies += "org.apache.spark" %% "spark-sql" % "2.4.5"

### **`Paso 6` - Copiar el archivo "_build.sbt_", "_plugins.sbt_" y el script scala "_script_spark.scala_" en el directorio recién creado**

In [ ]:
docker cp build.sbt spark-master:/hadoop-data/proyecto_3
docker cp script_spark.scala spark-master:/hadoop-data/proyecto_3/src/main/scala
docker cp plugins.sbt spark-master:/hadoop-data/proyecto_3/project

In [ ]:
C:\Users\Alfonso\Documents\ecosistema-hadoop\material>

C:\Users\Alfonso\Documents\ecosistema-hadoop\material>cd proyecto_3

C:\Users\Alfonso\Documents\ecosistema-hadoop\material\proyecto_3>docker cp build.sbt spark-master:/hadoop-data/proyecto_3
Successfully copied 2.05kB to spark-master:/hadoop-data/proyecto_3

C:\Users\Alfonso\Documents\ecosistema-hadoop\material\proyecto_3>docker cp script_spark.scala spark-master:/hadoop-data/proyecto_3/src/main/scala
Successfully copied 10.8kB to spark-master:/hadoop-data/proyecto_3/src/main/scala

C:\Users\Alfonso\Documents\ecosistema-hadoop\material\proyecto_3>docker cp plugins.sbt spark-master:/hadoop-data/proyecto_3/project
Successfully copied 2.05kB to spark-master:/hadoop-data/proyecto_3/project

C:\Users\Alfonso\Documents\ecosistema-hadoop\material\proyecto_3>docker exec -it spark-master bash
root@spark-master:/#
root@spark-master:/# cd /hadoop-data/proyecto_3
root@spark-master:/hadoop-data/proyecto_3#
root@spark-master:/hadoop-data/proyecto_3# ls -l
total 12
-rwxr-xr-x 1 root root  479 Jun 29 16:52 build.sbt
drwxr-xr-x 2 root root 4096 Jun 29 16:53 project
drwxr-xr-x 3 root root 4096 Jun 29 16:48 src
root@spark-master:/hadoop-data/proyecto_3#
root@spark-master:/hadoop-data/proyecto_3# cd project
root@spark-master:/hadoop-data/proyecto_3/project#
root@spark-master:/hadoop-data/proyecto_3/project# ls -l
total 4
-rwxr-xr-x 1 root root 91 Jun 29 16:52 plugins.sbt
root@spark-master:/hadoop-data/proyecto_3/project#
root@spark-master:/hadoop-data/proyecto_3/project# cd ..
root@spark-master:/hadoop-data/proyecto_3#
root@spark-master:/hadoop-data/proyecto_3# cd hadoop-data/proyecto_3/src/main/scala
root@spark-master:/hadoop-data/proyecto_3/src/main/scala#
root@spark-master:/hadoop-data/proyecto_3/src/main/scala# ls -l
total 1
-rwxr-xr-x 1 root root 8934 Jun 27 17:53 script_spark.scala
root@spark-master:/hadoop-data/proyecto_3/src/main/scala#

### **`Paso 7` - Compilar el archivo "_script_spark.scala_" en un archivo .jar utilizando sbt**

In [ ]:
sbt assembly

In [ ]:
C:\Users\Alfonso\Documents\ecosistema-hadoop\material\proyecto_3>docker exec -it spark-master bash
root@spark-master:/#
root@spark-master:/# cd hadoop-data/proyecto_3
root@spark-master:/hadoop-data/proyecto_3#
root@spark-master:/hadoop-data/proyecto_3# sbt assembly
[info] Updated file /hadoop-data/proyecto_3/project/build.properties: set sbt.version to 1.2.8
[info] Loading settings for project proyecto_3-build from plugins.sbt ...
[info] Loading project definition from /hadoop-data/proyecto_3/project
[info] Updating ProjectRef(uri("file:/hadoop-data/proyecto_3/project/"), "proyecto_3-build")...
[info] Done updating.
[warn] There may be incompatibilities among your library dependencies; run 'evicted' to see detailed eviction warnings.
[info] Loading settings for project proyecto_3 from build.sbt ...
[info] Set current project to etl-spark-scala (in build file:/hadoop-data/proyecto_3/)
[info] Updating ...
[info] downloading https://repo1.maven.org/maven2/org/apache/avro/avro/1.8.2/avro-1.8.2.jar ...
[info]  [SUCCESSFUL ] org.apache.avro#avro;1.8.2!avro.jar (3115ms)
[info] downloading https://repo1.maven.org/maven2/org/apache/avro/avro-ipc/1.8.2/avro-ipc-1.8.2.jar ...
[info]  [SUCCESSFUL ] org.apache.avro#avro-ipc;1.8.2!avro-ipc.jar (3705ms)
[info] Done updating.
[warn] There may be incompatibilities among your library dependencies; run 'evicted' to see detailed eviction warnings.
[info] Compiling 1 Scala source to /hadoop-data/proyecto_3/target/scala-2.12/classes ...
[info] Done compiling.
[info] Strategy 'discard' was applied to 334 files (Run the task at debug level to see details)
[info] Strategy 'first' was applied to 383 files (Run the task at debug level to see details)
[info] Packaging /hadoop-data/proyecto_3/target/scala-2.12/etl-spark-scala-assembly.jar ...
[info] Done packaging.
[success] Total time: 101 s, completed Jun 29, 2024 5:06:53 PM
root@spark-master:/hadoop-data/proyecto_3#
root@spark-master:/hadoop-data/proyecto_3# ls -l
total 16
-rwxr-xr-x 1 root root  612 Jun 29 17:04 build.sbt
drwxr-xr-x 4 root root 4096 Jun 29 17:05 project
drwxr-xr-x 3 root root 4096 Jun 29 17:03 src
drwxr-xr-x 4 root root 4096 Jun 29 17:05 target
root@spark-master:/hadoop-data/proyecto_3#
root@spark-master:/hadoop-data/proyecto_3# cd project
root@spark-master:/hadoop-data/proyecto_3/project#
root@spark-master:/hadoop-data/proyecto_3/project# ls -l
total 16
-rw-r--r-- 1 root root   18 Jun 29 17:04 build.properties
-rwxr-xr-x 1 root root   91 Jun 29 16:52 plugins.sbt
drwxr-xr-x 3 root root 4096 Jun 29 17:04 project
drwxr-xr-x 5 root root 4096 Jun 29 17:05 target
root@spark-master:/hadoop-data/proyecto_3/project#
root@spark-master:/hadoop-data/proyecto_3/project# cd project
root@spark-master:/hadoop-data/proyecto_3/project/project#
root@spark-master:/hadoop-data/proyecto_3/project/project# ls -l
total 4
drwxr-xr-x 3 root root 4096 Jun 29 17:04 target
root@spark-master:/hadoop-data/proyecto_3/project/project#
root@spark-master:/hadoop-data/proyecto_3/project/project# cd target
root@spark-master:/hadoop-data/proyecto_3/project/project/target#
root@spark-master:/hadoop-data/proyecto_3/project/project/target# ls -l
total 4
drwxr-xr-x 2 root root 4096 Jun 29 17:05 config-classes
root@spark-master:/hadoop-data/proyecto_3/project/project/target#
root@spark-master:/hadoop-data/proyecto_3/project/project/target# cd ..
root@spark-master:/hadoop-data/proyecto_3/project/project# cd ..
root@spark-master:/hadoop-data/proyecto_3/project# cd ..
root@spark-master:/hadoop-data/proyecto_3# cd target
root@spark-master:/hadoop-data/proyecto_3/target#
root@spark-master:/hadoop-data/proyecto_3/target# ls -l
total 8
drwxr-xr-x 4 root root 4096 Jun 29 17:05 scala-2.12
drwxr-xr-x 6 root root 4096 Jun 29 17:05 streams
root@spark-master:/hadoop-data/proyecto_3/target#
root@spark-master:/hadoop-data/proyecto_3/target# cd scala-2.12
root@spark-master:/hadoop-data/proyecto_3/target/scala-2.12#
root@spark-master:/hadoop-data/proyecto_3/target/scala-2.12# ls -l
total 98380
drwxr-xr-x 2 root root      4096 Jun 29 17:05 classes
-rw-r--r-- 1 root root 100729530 Jun 29 17:06 etl-spark-scala-assembly.jar
drwxr-xr-x 4 root root      4096 Jun 29 17:05 resolution-cache
root@spark-master:/hadoop-data/proyecto_3/target/scala-2.12#
root@spark-master:/hadoop-data/proyecto_3/target/scala-2.12#

### **`Paso 8` - Despliegue de archivo .jar de procesamiento SPARK**

In [ ]:
spark-submit --class Main /hadoop-data/proyecto_3/target/scala-2.12/etl-spark-scala-assembly.jar

### **`Paso 9` - Revisión de resultados en HDFS**

In [ ]:
C:\Users\Alfonso\Documents\ecosistema-hadoop\material\proyecto_3>docker exec -it namenode bash
root@namenode:/#
root@namenode:/# hdfs dfs -ls /user/proyectos/proyecto_3
Found 2 items
drwxr-xr-x   - root supergroup          0 2024-06-29 16:16 /user/proyectos/proyecto_3/input
drwxr-xr-x   - root supergroup          0 2024-06-29 17:10 /user/proyectos/proyecto_3/output
root@namenode:/#
root@namenode:/# hdfs dfs -ls /user/proyectos/proyecto_3/input
Found 2 items
-rw-r--r--   3 root supergroup       1658 2024-06-29 16:16 /user/proyectos/proyecto_3/input/RIESGO_CREDITICIO.csv
-rw-r--r--   3 root supergroup    4491352 2024-06-29 16:16 /user/proyectos/proyecto_3/input/transacciones_bancarias.json
root@namenode:/#
root@namenode:/# hdfs dfs -ls /user/proyectos/proyecto_3/output
Found 3 items
drwxr-xr-x   - root supergroup          0 2024-06-29 17:10 /user/proyectos/proyecto_3/output/REPORTE_1
drwxr-xr-x   - root supergroup          0 2024-06-29 17:10 /user/proyectos/proyecto_3/output/REPORTE_2
drwxr-xr-x   - root supergroup          0 2024-06-29 17:11 /user/proyectos/proyecto_3/output/REPORTE_3
root@namenode:/#
root@namenode:/# hdfs dfs -ls /user/proyectos/proyecto_3/output/REPORTE_1
Found 3 items
-rw-r--r--   3 root supergroup          0 2024-06-29 17:10 /user/proyectos/proyecto_3/output/REPORTE_1/_SUCCESS
-rw-r--r--   3 root supergroup       1459 2024-06-29 17:10 /user/proyectos/proyecto_3/output/REPORTE_1/part-00000-26d29525-a3ba-4461-a699-8b65bf160e8a-c000.csv
-rw-r--r--   3 root supergroup        235 2024-06-29 17:10 /user/proyectos/proyecto_3/output/REPORTE_1/part-00001-26d29525-a3ba-4461-a699-8b65bf160e8a-c000.csv
root@namenode:/#
root@namenode:/# hdfs dfs -ls /user/proyectos/proyecto_3/output/REPORTE_2
Found 3 items
-rw-r--r--   3 root supergroup          0 2024-06-29 17:10 /user/proyectos/proyecto_3/output/REPORTE_2/_SUCCESS
-rw-r--r--   3 root supergroup       2925 2024-06-29 17:10 /user/proyectos/proyecto_3/output/REPORTE_2/part-00000-6fb43c28-53f1-4250-90d5-c97efe53e7e4-c000.csv
-rw-r--r--   3 root supergroup        251 2024-06-29 17:10 /user/proyectos/proyecto_3/output/REPORTE_2/part-00001-6fb43c28-53f1-4250-90d5-c97efe53e7e4-c000.csv
root@namenode:/#
root@namenode:/# hdfs dfs -ls /user/proyectos/proyecto_3/output/REPORTE_3
Found 3 items
-rw-r--r--   3 root supergroup          0 2024-06-29 17:11 /user/proyectos/proyecto_3/output/REPORTE_3/_SUCCESS
-rw-r--r--   3 root supergroup       1103 2024-06-29 17:11 /user/proyectos/proyecto_3/output/REPORTE_3/part-00000-3cb62c9a-7a12-4bea-8820-f80ca9fd34db-c000.csv
-rw-r--r--   3 root supergroup        185 2024-06-29 17:11 /user/proyectos/proyecto_3/output/REPORTE_3/part-00001-3cb62c9a-7a12-4bea-8820-f80ca9fd34db-c000.csv
root@namenode:/#